In [15]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

# Load the dataset
newsgroups = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

# Preprocess the text
vectorizer = CountVectorizer(
    stop_words="english",
    max_features=1000
)

X = vectorizer.fit_transform(newsgroups.data)

# Vocabulary and word counts
feature_names = vectorizer.get_feature_names_out()
word_counts = np.asarray(X.sum(axis=0)).flatten()
total_words = word_counts.sum()
K = len(feature_names)

# -------------------- MLE --------------------
mle_probs = word_counts / total_words

# -------------------- MAP --------------------
alphas = [0.5, 1.0, 2.0]
map_probs = {}

for alpha in alphas:
    map_probs[alpha] = (word_counts + alpha - 1) / (
        total_words + K * (alpha - 1)
    )

# Compare first 10 words
comparison = pd.DataFrame({
    "Word": feature_names[:10],
    "MLE": mle_probs[:10],
    "MAP (α=0.5)": map_probs[0.5][:10],
    "MAP (α=1.0)": map_probs[1.0][:10],
    "MAP (α=2.0)": map_probs[2.0][:10]
})

print("Comparison of MLE and MAP Estimates")
print(comparison)

# Effect of different priors
print("\nAverage Difference from MLE")

for alpha in alphas:
    difference = np.mean(np.abs(mle_probs - map_probs[alpha]))
    print(f"α = {alpha}: {difference:.8f}")

Comparison of MLE and MAP Estimates
  Word       MLE  MAP (α=0.5)  MAP (α=1.0)  MAP (α=2.0)
0   00  0.002188     0.002189     0.002188     0.002186
1  000  0.001259     0.001259     0.001259     0.001259
2   01  0.000414     0.000414     0.000414     0.000415
3   02  0.000476     0.000476     0.000476     0.000477
4   03  0.000383     0.000383     0.000383     0.000385
5   04  0.000632     0.000632     0.000632     0.000633
6   0d  0.000469     0.000468     0.000469     0.000470
7   0t  0.000918     0.000918     0.000918     0.000918
8   10  0.003031     0.003033     0.003031     0.003027
9  100  0.000979     0.000979     0.000979     0.000979

Average Difference from MLE
α = 0.5: 0.00000058
α = 1.0: 0.00000000
α = 2.0: 0.00000116
